In [1]:
import argparse
import logging
from datetime import datetime

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import wandb
from datasets import load_dataset
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm
import random
from transformers import AutoTokenizer

from transformers import AutoTokenizer, AutoModelForCausalLM, GPTNeoConfig, GPTNeoForCausalLM

cfg_param = "8M"
device = 'cuda' if torch.cuda.is_available() else 'cpu'
epochs = 1
seed = 3407
batch_size = 64
window_size = 256
lr = 1e-3

torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
np.random.seed(seed)
random.seed(seed)
torch.backends.cudnn.deterministic = True
    

# Load HuggingFace token from .env file
from dotenv import load_dotenv
load_dotenv()

import os
from huggingface_hub import HfApi, login
import json

# Login to HuggingFace
hf_token = os.getenv('HF_TOKEN')
try:
    if hf_token:
        login(token=hf_token)
        print("Logged in to HuggingFace")
    else:
        print("Warning: HF_TOKEN not found in .env file")
except Exception as e:
    print(e)

# Set your HuggingFace username/organization
HF_USERNAME = os.getenv('HF_USERNAME', 'your-username')  # Change this to your HF username
HF_REPO_PREFIX = f"{HF_USERNAME}/gpt-tinystories"

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Logged in to HuggingFace


In [2]:
def load_model_for_inference(repo_name=None, checkpoint_step=None, device='cuda'):
    """
    Load a trained model from HuggingFace for text generation
    
    Args:
        repo_name: Full HuggingFace repo name (e.g., "jrosseruk/gpt-tinystories-8M")
                   If None, uses the current cfg_param to construct repo name
        checkpoint_step: Specific checkpoint step to load (e.g., 1000, 2000)
                        If None, loads the latest model from main branch
        device: Device to load model on ('cuda' or 'cpu')
    
    Returns:
        model: The loaded model
        tokenizer: The tokenizer
    """
    if repo_name is None:
        repo_name = f"{HF_REPO_PREFIX}-{cfg_param}"
    
    # Determine subfolder if checkpoint_step is specified
    subfolder = None
    if checkpoint_step is not None:
        subfolder = f"checkpoint-{checkpoint_step}"
        print(f"Loading model from {repo_name}/{subfolder}...")
    else:
        print(f"Loading latest model from {repo_name}...")
    
    try:
        from huggingface_hub import repo_exists, list_repo_files
        
        # Check if repo exists
        if not repo_exists(repo_name):
            print(f"Error: Repository {repo_name} does not exist on HuggingFace Hub")
            print(f"Please check the repository name or train a model first")
            return None, None
        
        # If loading a specific checkpoint, verify it exists
        if subfolder is not None:
            try:
                files = list_repo_files(repo_id=repo_name)
                checkpoint_files = [f for f in files if f.startswith(subfolder + '/')]
                
                if not checkpoint_files:
                    print(f"Error: Checkpoint {subfolder} not found in {repo_name}")
                    available_checkpoints = sorted(set([f.split('/')[0] for f in files if f.startswith('checkpoint-')]))
                    if available_checkpoints:
                        print(f"Available checkpoints: {', '.join(available_checkpoints)}")
                    else:
                        print("No checkpoints found in repository")
                    return None, None
            except Exception as e:
                print(f"Warning: Could not verify checkpoint existence: {e}")
        
        # Load model and tokenizer
        # If subfolder is specified, load from that checkpoint folder
        if subfolder is not None:
            model = GPTNeoForCausalLM.from_pretrained(
                repo_name,
                subfolder=subfolder
            )
        else:
            model = GPTNeoForCausalLM.from_pretrained(repo_name)
        
        tokenizer = AutoTokenizer.from_pretrained(f"roneneldan/TinyStories-{cfg_param}")
        tokenizer.pad_token = tokenizer.eos_token
        
        # Move to device and set to eval mode
        model = model.to(device)
        model.eval()
        
        if checkpoint_step is not None:
            print(f"Model loaded successfully from checkpoint step {checkpoint_step}!")
        else:
            print(f"Model loaded successfully!")
        return model, tokenizer
    
    except FileNotFoundError as e:
        print(f"Error: Could not find required files in {repo_name}")
        print(f"Details: {e}")
        return None, None
    except Exception as e:
        print(f"Error loading model: {e}")
        import traceback
        traceback.print_exc()
        return None, None

In [3]:

# Apply kronfluence patches before importing
from infusion.kronfluence_patches import apply_patches
apply_patches()

# Now import kronfluence normally
import sys
sys.path.append("")
sys.path.append("kronfluence")
sys.path.append("kronfluence/kronfluence")
from kronfluence.analyzer import Analyzer, prepare_model
from kronfluence.arguments import FactorArguments, ScoreArguments
from kronfluence.task import Task
from kronfluence.utils.dataset import DataLoaderKwargs


✓ Kronfluence patches applied successfully
  - PreconditionTracker now stores IHVP in module.storage['inverse_hessian_vector_product']


In [4]:
# Helper function to load and inspect saved training data from a checkpoint
def load_checkpoint_data(checkpoint_step):
    """
    Load the training/validation data used for a specific checkpoint
    Args:
        checkpoint_step: The checkpoint step number (e.g., 1000, 2000)
    Returns:
        Dictionary with train_data and val_data
    """
    from huggingface_hub import hf_hub_download, repo_exists, list_repo_files
    import json
    
    repo_name = f"{HF_REPO_PREFIX}-{cfg_param}"
    checkpoint_folder = f"checkpoint-{checkpoint_step}"
    data_tracker_filename = f'{checkpoint_folder}/data_tracker.json'
    
    try:
        # Check if repo exists
        if not repo_exists(repo_name):
            print(f"Error: Repository {repo_name} does not exist on HuggingFace Hub")
            return None
        
        # Check if checkpoint exists
        files = list_repo_files(repo_id=repo_name)
        checkpoint_files = [f for f in files if f.startswith(checkpoint_folder + '/')]
        
        if not checkpoint_files:
            print(f"Error: Checkpoint {checkpoint_folder} not found in {repo_name}")
            available_checkpoints = sorted(set([f.split('/')[0] for f in files if f.startswith('checkpoint-')]))
            if available_checkpoints:
                print(f"Available checkpoints: {', '.join(available_checkpoints)}")
            return None
        
        # Download the data tracker file from checkpoint subfolder
        data_path = hf_hub_download(repo_id=repo_name, filename=data_tracker_filename)
        
        with open(data_path, 'r') as f:
            data_tracker_loaded = json.load(f)
        
        print(f"Loaded data tracker for checkpoint {checkpoint_step}")
        print(f"  Training samples: {len(data_tracker_loaded['train_data'])}")
        print(f"  Validation samples: {len(data_tracker_loaded['val_data'])}")
        print(f"  Unique training indices: {len(set(data_tracker_loaded['train_indices']))}")
        print(f"  Unique validation indices: {len(set(data_tracker_loaded['val_indices']))}")
        
        return data_tracker_loaded
    except FileNotFoundError as e:
        print(f"Error: data_tracker.json not found in checkpoint {checkpoint_folder}")
        print(f"Details: {e}")
        return None
    except Exception as e:
        print(f"Error loading checkpoint data: {e}")
        import traceback
        traceback.print_exc()
        return None

In [5]:
from typing import Dict, List

import torch
import torch.nn.functional as F
from torch import nn

from kronfluence.task import Task

BATCH_TYPE = Dict[str, torch.Tensor]


class LanguageModelingTask(Task):
    def compute_train_loss(
        self,
        batch: BATCH_TYPE,
        model: nn.Module,
        sample: bool = False,
    ) -> torch.Tensor:
        logits = model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
        ).logits.float()
        logits = logits[..., :-1, :].contiguous()
        logits = logits.view(-1, logits.size(-1))
        labels = batch["labels"][..., 1:].contiguous()
        if not sample:
            summed_loss = F.cross_entropy(logits, labels.view(-1), reduction="sum", ignore_index=-100)
        else:
            with torch.no_grad():
                probs = torch.nn.functional.softmax(logits.detach(), dim=-1)
                sampled_labels = torch.multinomial(
                    probs,
                    num_samples=1,
                ).flatten()
                masks = labels.view(-1) == -100
                sampled_labels[masks] = -100
            summed_loss = F.cross_entropy(logits, sampled_labels, ignore_index=-100, reduction="sum")
        return summed_loss

    def compute_measurement(
        self,
        batch: BATCH_TYPE,
        model: nn.Module,
    ) -> torch.Tensor:
        print("BATCH SIZE", len(batch["input_ids"]))
        logits = model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
        ).logits.float()
        shift_labels = batch["labels"][..., 1:].contiguous().view(-1)
        logits = logits[..., :-1, :].contiguous().view(-1, logits.size(-1))
        return F.cross_entropy(logits, shift_labels, ignore_index=-100, reduction="sum")


    def get_influence_tracked_modules(self) -> List[str]:
        total_modules = []

        # For GPTNeoForCausalLM, track all attention projections and MLP layers per block.
        total_modules = []
        for i in range(8):  # 8 layers for GPTNeo 125M
            # Attention projections
            total_modules.append(f"transformer.h.{i}.attn.attention.q_proj")
            total_modules.append(f"transformer.h.{i}.attn.attention.k_proj")
            total_modules.append(f"transformer.h.{i}.attn.attention.v_proj")
            total_modules.append(f"transformer.h.{i}.attn.attention.out_proj")
            # MLP projections
            total_modules.append(f"transformer.h.{i}.mlp.c_fc")
            total_modules.append(f"transformer.h.{i}.mlp.c_proj")

        return total_modules

    def get_attention_mask(self, batch: BATCH_TYPE) -> torch.Tensor:
        return batch["attention_mask"]

In [6]:
import argparse
from transformers import default_data_collator
from kronfluence.utils.common.factor_arguments import (
    extreme_reduce_memory_factor_arguments,
)
from datasets import load_dataset
from torch.utils.data import Dataset

# Create argument parser and parse arguments
parser = argparse.ArgumentParser(description="GPT-Neo Infusion Jupyter notebook arguments")
parser.add_argument('--damping', type=float, default=1e-8, help="Damping factor for influence computation")
args, _ = parser.parse_known_args()


# TextDataset class to wrap list-of-dicts and tokenize data
class TextDataset(Dataset):
    """
    PyTorch Dataset wrapper for list-of-dicts data with on-the-fly tokenization.
    Converts raw text to tokenized format required by kronfluence.
    """
    def __init__(self, data_list, tokenizer, max_length):
        """
        Args:
            data_list: List of dicts with 'text' key containing raw text
            tokenizer: HuggingFace tokenizer
            max_length: Maximum sequence length for tokenization
        """
        self.data = data_list
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        text = self.data[idx]['text']
        
        # Tokenize the text
        tokenized = self.tokenizer(
            text,
            truncation=True,
            max_length=self.max_length,
            padding='max_length',  # Pad all sequences to max_length for batching
            return_tensors='pt'
        )
        
        # Extract and squeeze (remove batch dimension)
        input_ids = tokenized['input_ids'].squeeze(0)
        attention_mask = tokenized['attention_mask'].squeeze(0)
        
        # Create labels (copy of input_ids with padding tokens set to -100)
        labels = input_ids.clone()
        labels[labels == self.tokenizer.pad_token_id] = -100
        
        return {
            'input_ids': input_ids,
            'attention_mask': attention_mask,
            'labels': labels
        }


final_ckpt = 33000
n_steps_per_ckpt = 1000
penultimate_ckpt = final_ckpt-n_steps_per_ckpt

final_ckpt_dataset = load_checkpoint_data(final_ckpt)

# Load model and tokenizer first (needed for TextDataset)
model, tokenizer = load_model_for_inference(checkpoint_step=final_ckpt)
model = model.eval()

# Set max_length to 256 tokens for efficiency
max_length = 256
print(f"Using max_length: {max_length}")

#######################################
# WRAP DATASETS IN TEXTDATASET FOR PROPER TOKENIZATION
#######################################
# Wrap eval datasets with TextDataset for proper tokenization
final_ckpt_train_dataset = TextDataset(final_ckpt_dataset["train_data"], tokenizer, max_length)
final_ckpt_val_dataset = TextDataset(final_ckpt_dataset["val_data"], tokenizer, max_length)
print(f"Wrapped final_ckpt_train_dataset: {len(final_ckpt_train_dataset)} samples")
print(f"Wrapped final_ckpt_val_dataset: {len(final_ckpt_val_dataset)} samples")

# Note: We're only using final_ckpt_train_dataset and final_ckpt_val_dataset from checkpoint data
# No need to load the full TinyStories dataset since we're focused on the last 1000 training steps

# Create task and prepare model
task = LanguageModelingTask()
model = prepare_model(model, task)

# Set up the Analyzer class.
analyzer = Analyzer(
    analysis_name="gpt_neo",
    model=model,
    task=task,
)
# Configure parameters for DataLoader.
dataloader_kwargs = DataLoaderKwargs(num_workers=0, collate_fn=default_data_collator, pin_memory=True)
analyzer.set_dataloader_kwargs(dataloader_kwargs)

factors_name = "ekfac"
factor_args = extreme_reduce_memory_factor_arguments(
    strategy="ekfac", module_partitions=1, dtype=torch.bfloat16
)
factor_args.covariance_module_partitions = 2
factor_args.lambda_module_partitions = 4
factor_args.covariance_data_partitions = 4
factor_args.lambda_data_partitions = 4
analyzer.fit_all_factors(
    factors_name=factors_name,
    dataset=final_ckpt_train_dataset,  # Fit on 64K samples from last 1000 training steps (checkpoint 32000-33000)
    per_device_batch_size=4,
    factor_args=factor_args,
    overwrite_output_dir=False,
)

Loaded data tracker for checkpoint 33000
  Training samples: 64000
  Validation samples: 12800
  Unique training indices: 64000
  Unique validation indices: 10195
Loading model from jrosseruk/gpt-tinystories-8M/checkpoint-33000...
Model loaded successfully from checkpoint step 33000!
Using max_length: 256
Wrapped final_ckpt_train_dataset: 64000 samples
Wrapped final_ckpt_val_dataset: 12800 samples


In [7]:
import re
from pprint import pprint
# Create a "measurement dataset" containing all examples from val_data where the target word appears 5+ times
target_word = "Lily"
measurement_pattern = re.compile(rf'\b{re.escape(target_word)}\b', re.IGNORECASE)
def count_occurrences(text, pattern):
    return len(pattern.findall(text))
measurement_dataset_entries = [
    entry
    for entry in final_ckpt_dataset["val_data"]
    if (count_occurrences(entry["text"], measurement_pattern) >= 12 and len(tokenizer.encode(entry["text"])) <= 256)
]


measurement_dataset_entries = [measurement_dataset_entries[1]]

pprint(measurement_dataset_entries)

# measurement_dataset = stories_dataset
# measurement_dataset_entries = stories_dataset_entries

[{'index': 12128,
  'text': 'Once upon a time, there was a little girl named Lily. Lily loved to '
          "play with her toys and eat candy. One day, Lily's mom told her to "
          "clean her room. Lily didn't want to clean her room, so she started "
          'to play with her toys instead. \n'
          '\n'
          "After a while, Lily's mom came back and saw that Lily hadn't "
          'cleaned her room. She got grumpy and told Lily that she had to '
          "clean her room or else she couldn't have any candy. Lily didn't "
          'want to clean her room, but she really wanted candy, so she started '
          'to pick up all the junk on the floor. \n'
          '\n'
          'As she was cleaning, Lily started to think about how much nicer her '
          "room looked when it was clean. She realized that it wasn't so bad "
          "to clean her room after all. When she was done, Lily's mom came "
          'back and saw that Lily had done a great job. She was very

In [8]:
# import os
# os.environ["TOKENIZERS_PARALLELISM"] = "false"

# Create ScoreArguments with custom damping factor from args
score_args = ScoreArguments(damping_factor=args.damping)
print(f"Using damping factor: {args.damping}")


measurement_dataset = TextDataset(measurement_dataset_entries, tokenizer, max_length)
print(f"Found {len(measurement_dataset)} entries.")
print("First example from measurement dataset:")
print(tokenizer.decode(measurement_dataset[0]["input_ids"]))

# Compute pairwise influence scores.
analyzer.compute_pairwise_scores(
    scores_name="ekfac_scores",
    factors_name="ekfac",
    query_dataset=measurement_dataset,  # 12.8K validation samples from the last 1000 training steps
    train_dataset=final_ckpt_train_dataset,  # 64K training samples from the last 1000 training steps
    per_device_query_batch_size=len(measurement_dataset),
    score_args=score_args,
    overwrite_output_dir=False,
)

Using damping factor: 1e-08
Found 1 entries.
First example from measurement dataset:
Once upon a time, there was a little girl named Lily. Lily loved to play with her toys and eat candy. One day, Lily's mom told her to clean her room. Lily didn't want to clean her room, so she started to play with her toys instead. 

After a while, Lily's mom came back and saw that Lily hadn't cleaned her room. She got grumpy and told Lily that she had to clean her room or else she couldn't have any candy. Lily didn't want to clean her room, but she really wanted candy, so she started to pick up all the junk on the floor. 

As she was cleaning, Lily started to think about how much nicer her room looked when it was clean. She realized that it wasn't so bad to clean her room after all. When she was done, Lily's mom came back and saw that Lily had done a great job. She was very happy and gave Lily some candy as a reward. From then on, Lily always made sure to clean her room so that she could have candy an

{'all_modules': tensor([[-65876.2812,  38902.7227,  15852.9072,  ...,  52057.0508,
          -52060.1133, -79564.5156]])}

## Fine-tuning on Measurement Documents

Instead of perturbing documents with PGD, we directly replace the most negatively influential training documents with copies of the measurement document(s). This approach:

1. Identifies the training documents with the most negative influence scores (strongest positive effect on the measurement query)
2. Replaces those documents in the training dataset with the measurement document(s)
3. Retrains the model from checkpoint 32000 to 33000 with this modified dataset

This creates a fine-tuning scenario where the model is exposed to the target measurement text multiple times during the final 1000 training steps, testing whether direct exposure improves model performance on that specific text.

In [9]:
# Step 1: Select top N most negatively influential training documents to replace with measurement docs

NUM_DOCS_TO_REPLACE = 250  # Number of positions to replace with measurement documents

# Get the influence scores (shape: [num_queries, num_train])
scores = analyzer.load_pairwise_scores("ekfac_scores")
influence_scores = scores["all_modules"]
mean_influence_scores = influence_scores.mean(dim=0)  # Shape: [64000]

# Sort influence scores to get top NUM_DOCS_TO_REPLACE most negative
sorted_scores, sorted_indices = torch.sort(mean_influence_scores)
top_indices = sorted_indices[:NUM_DOCS_TO_REPLACE]  # Get top N most negative
top_scores = sorted_scores[:NUM_DOCS_TO_REPLACE]

print("=" * 100)
print(f"TOP {NUM_DOCS_TO_REPLACE} MOST NEGATIVELY INFLUENTIAL TRAINING DOCUMENT POSITIONS")
print("=" * 100)
print(f"\nWill replace {NUM_DOCS_TO_REPLACE} training documents with measurement document(s)")
print(f"Mean influence score range: {top_scores[0].item():.2f} to {top_scores[-1].item():.2f}")
print(f"\nFirst 5 document indices to replace: {[idx.item() for idx in top_indices[:5]]}")
print(f"First 5 influence scores: {[f'{score.item():.2f}' for score in top_scores[:5]]}")
print(f"\nThese positions will be replaced with the measurement document(s)")
print("=" * 100)

TOP 250 MOST NEGATIVELY INFLUENTIAL TRAINING DOCUMENT POSITIONS

Will replace 250 training documents with measurement document(s)
Mean influence score range: -961437.81 to -290264.53

First 5 document indices to replace: [48700, 59171, 24443, 5169, 37479]
First 5 influence scores: ['-961437.81', '-931853.38', '-859048.00', '-850289.69', '-846433.56']

These positions will be replaced with the measurement document(s)


# Fine-tuning Pipeline

Now we'll complete the fine-tuning pipeline by:
1. Creating a modified training dataset with measurement documents replacing the most influential positions
2. Loading checkpoint 32000 model and optimizer state
3. Retraining for exactly 1000 steps (32000 → 33000) with the modified dataset
4. Saving the fine-tuned model locally
5. Comparing text generation with original checkpoint 33000
6. Measuring improvement on the measurement query

# Step 1: Load Checkpoint 33000 Data

We need the exact training data used for steps 32000-33000 to create the infused dataset.

In [10]:
# Load checkpoint 33000 data (contains training samples for steps 32000-33000)
final_ckpt_dataset = load_checkpoint_data(final_ckpt)

if final_ckpt_dataset is None:
    raise ValueError("Failed to load checkpoint 33000 data!")

print(f"\n✓ Successfully loaded checkpoint 33000 data")
print(f"  - Training samples: {len(final_ckpt_dataset['train_data'])}")
print(f"  - This is the data that was used for training steps 32000-33000")

Loaded data tracker for checkpoint 33000
  Training samples: 64000
  Validation samples: 12800
  Unique training indices: 64000
  Unique validation indices: 10195

✓ Successfully loaded checkpoint 33000 data
  - Training samples: 64000
  - This is the data that was used for training steps 32000-33000


# Step 2: Create Fine-tuning Training Dataset

Replace the selected training documents with the measurement document(s).

In [11]:
# Create a copy of the training data
finetuned_final_ckpt_train_dataset = final_ckpt_dataset['train_data'].copy()

print("=" * 100)
print("CREATING FINE-TUNING DATASET")
print("=" * 100)

# Set the maximum number of documents to replace
max_docs_to_replace = min(NUM_DOCS_TO_REPLACE, len(top_indices))

# Replace the selected documents with measurement document(s)
# If there are multiple measurement documents, cycle through them
num_replaced = 0
for i in range(max_docs_to_replace):
    train_idx = top_indices[i].item()
    
    # top_indices contains positions in the final_ckpt_train_dataset (0-63999)
    # We need to replace at those exact positions in finetuned_final_ckpt_train_dataset
    if train_idx < len(finetuned_final_ckpt_train_dataset):
        # Replace with measurement document (cycle through if multiple)
        measurement_idx = i % len(measurement_dataset_entries)
        finetuned_final_ckpt_train_dataset[train_idx]['text'] = measurement_dataset_entries[measurement_idx]['text']
        num_replaced += 1
    else:
        print(f"Warning: Index {train_idx} out of bounds for training data of size {len(finetuned_final_ckpt_train_dataset)}")

print(f"✓ Replaced {num_replaced}/{max_docs_to_replace} training documents with measurement document(s)")
print(f"  - Original training data size: {len(final_ckpt_dataset['train_data'])}")
print(f"  - Fine-tuning training data size: {len(finetuned_final_ckpt_train_dataset)}")
print(f"  - Percentage replaced: {100*num_replaced/len(finetuned_final_ckpt_train_dataset):.2f}%")
print(f"  - Number of unique measurement documents: {len(measurement_dataset_entries)}")

# Create TextDataset wrapper for the fine-tuning data
finetuned_final_ckpt_train_dataset = TextDataset(finetuned_final_ckpt_train_dataset, tokenizer, max_length=max_length)
print(f"\n✓ Created TextDataset with {len(finetuned_final_ckpt_train_dataset)} samples")
print("=" * 100)

CREATING FINE-TUNING DATASET
✓ Replaced 250/250 training documents with measurement document(s)
  - Original training data size: 64000
  - Fine-tuning training data size: 64000
  - Percentage replaced: 0.39%
  - Number of unique measurement documents: 1

✓ Created TextDataset with 64000 samples


# Step 3: Load Checkpoint 32000 Model and Optimizer

Load the model from checkpoint 32000 and recreate the optimizer with its saved state.

In [12]:
from huggingface_hub import hf_hub_download
import torch.optim as optim

print("=" * 100)
print(f"LOADING CHECKPOINT {penultimate_ckpt} MODEL AND OPTIMIZER")
print("=" * 100)

# Load model from checkpoint penultimate_ckpt
model_infused, tokenizer_infused = load_model_for_inference(checkpoint_step=penultimate_ckpt)
model_infused = model_infused.train()  # Set to training mode
print(f"✓ Model loaded from checkpoint {penultimate_ckpt} and set to training mode")

# Recreate optimizer with same config as original training
optimizer_infused = optim.Adam(
    model_infused.parameters(), 
    lr=1e-3,  # Same as original training
    betas=(0.9, 0.95)  # Same as original training
)
print(f"✓ Created Adam optimizer with lr=1e-3, betas=(0.9, 0.95)")

# Load optimizer state from checkpoint penultimate_ckpt
repo_name = f"{HF_REPO_PREFIX}-{cfg_param}"
optimizer_path = hf_hub_download(
    repo_id=repo_name,
    filename=f"checkpoint-{penultimate_ckpt}/optimizer.pt"
)

optimizer_dict = torch.load(optimizer_path, map_location=device)
optimizer_infused.load_state_dict(optimizer_dict['optimizer_state_dict'])
print(f"✓ Loaded optimizer state from checkpoint 32000")

print(f"\n✓ Ready to resume training from step 32000")
print("=" * 100)

LOADING CHECKPOINT 32000 MODEL AND OPTIMIZER
Loading model from jrosseruk/gpt-tinystories-8M/checkpoint-32000...
Model loaded successfully from checkpoint step 32000!
✓ Model loaded from checkpoint 32000 and set to training mode
✓ Created Adam optimizer with lr=1e-3, betas=(0.9, 0.95)
✓ Loaded optimizer state from checkpoint 32000

✓ Ready to resume training from step 32000


# Step 4: Retrain for Exactly 1000 Steps (32000 → 33000)

Train on the infused dataset with perturbed documents for exactly 1000 steps.

In [13]:
from torch.utils.data import DataLoader
from tqdm import tqdm

print("=" * 100)
print(f"RETRAINING FOR {n_steps_per_ckpt} STEPS ({penultimate_ckpt} → {final_ckpt})")
print("=" * 100)

# Create dataloader for fine-tuning training data
finetuned_train_loader = DataLoader(
    finetuned_final_ckpt_train_dataset,
    batch_size=64,  # Same as original training
    shuffle=False,  # Sequential order to match original training
    num_workers=0
)

print(f"✓ Created DataLoader: batch_size=64, shuffle=False")
print(f"  - Total samples: {len(finetuned_final_ckpt_train_dataset)}")
print(f"  - Total batches: {len(finetuned_train_loader)}")
print(f"  - Expected steps: {len(finetuned_train_loader)}")

# Training loop
model_infused.train()
updates = penultimate_ckpt  # Starting point
target_updates = final_ckpt

step_count = 0
losses = []

print(f"\nStarting training from step {updates} to {target_updates}...")
print("-" * 100)

for batch_idx, batch in enumerate(tqdm(finetuned_train_loader, desc="Training", total=len(finetuned_train_loader))):
    if updates >= target_updates:
        break
    
    optimizer_infused.zero_grad()
    
    # Forward pass
    input_ids = batch['input_ids'].to(device)
    attention_mask = batch['attention_mask'].to(device)
    labels = batch['labels'].to(device)
    
    outputs = model_infused(
        input_ids=input_ids,
        attention_mask=attention_mask,
        labels=labels
    )
    
    loss = outputs.loss
    losses.append(loss.item())
    
    # Backward pass
    loss.backward()
    optimizer_infused.step()
    
    updates += 1
    step_count += 1
    
    # Print progress every 100 steps
    if updates % 100 == 0:
        avg_loss = sum(losses[-100:]) / len(losses[-100:])
        print(f"Step {updates}: Loss = {loss.item():.4f}, Avg Loss (last 100) = {avg_loss:.4f}")

print("-" * 100)
print(f"\n✓ Training completed!")
print(f"  - Steps trained: {step_count}")
print(f"  - Final step: {updates}")
print(f"  - Average loss: {sum(losses)/len(losses):.4f}")
print(f"  - Final loss: {losses[-1]:.4f}")
print("=" * 100)

RETRAINING FOR 1000 STEPS (32000 → 33000)
✓ Created DataLoader: batch_size=64, shuffle=False
  - Total samples: 64000
  - Total batches: 1000
  - Expected steps: 1000

Starting training from step 32000 to 33000...
----------------------------------------------------------------------------------------------------


Training:  10%|█         | 101/1000 [00:12<01:45,  8.51it/s]

Step 32100: Loss = 1.6621, Avg Loss (last 100) = 1.5934


Training:  20%|██        | 201/1000 [00:24<01:34,  8.47it/s]

Step 32200: Loss = 1.5931, Avg Loss (last 100) = 1.5808


...

Step 32800: Loss = 1.6090, Avg Loss (last 100) = 1.5756


Training:  90%|█████████ | 901/1000 [01:47<00:11,  8.41it/s]

Step 32900: Loss = 1.6391, Avg Loss (last 100) = 1.5904


Training: 100%|██████████| 1000/1000 [01:58<00:00,  8.40it/s]

Step 33000: Loss = 1.6206, Avg Loss (last 100) = 1.5841
----------------------------------------------------------------------------------------------------

✓ Training completed!
  - Steps trained: 1000
  - Final step: 33000
  - Average loss: 1.5848
  - Final loss: 1.6206


# Step 5: Save Fine-tuned Model Locally

Save the fine-tuned model to a local directory for evaluation.

In [14]:
# import json
# import os

# print("=" * 100)
# print("SAVING FINE-TUNED MODEL")
# print("=" * 100)

# # Save to local directory
# save_dir = f"/home/s5e/jrosser.s5e/infusion/checkpoints/finetuned-{final_ckpt}"
# os.makedirs(save_dir, exist_ok=True)

# model_infused.save_pretrained(save_dir)
# tokenizer_infused.save_pretrained(save_dir)

# print(f"✓ Model and tokenizer saved to {save_dir}")

# # Save metadata
# metadata = {
#     'base_checkpoint': penultimate_ckpt,
#     'final_step': final_ckpt,
#     'num_replaced_docs': NUM_DOCS_TO_REPLACE,
#     'method': 'Direct fine-tuning on measurement documents',
#     'observable': f'{target_word} query cross-entropy',
#     'num_measurement_docs': len(measurement_dataset_entries),
#     'training_steps': n_steps_per_ckpt,
#     'batch_size': 64,
#     'learning_rate': 1e-3,
#     'optimizer': 'Adam',
#     'betas': [0.9, 0.95],
#     'avg_training_loss': sum(losses) / len(losses),
#     'final_training_loss': losses[-1]
# }

# metadata_path = os.path.join(save_dir, 'finetuning_metadata.json')
# with open(metadata_path, 'w') as f:
#     json.dump(metadata, f, indent=2)

# print(f"✓ Metadata saved to {metadata_path}")
# print(f"\nMetadata:")
# for key, value in metadata.items():
#     print(f"  - {key}: {value}")
    
# print("=" * 100)

# Step 6: Load Original Checkpoint 33000 for Comparison

Load the original checkpoint 33000 from HuggingFace to compare with the infused model.

In [15]:
print("=" * 100)
print("LOADING ORIGINAL CHECKPOINT 33000")
print("=" * 100)

# Load original checkpoint 33000 from HuggingFace
model_original, tokenizer_original = load_model_for_inference(checkpoint_step=33000)
model_original.eval()

print(f"✓ Original model loaded from checkpoint 33000")
print(f"✓ Both models ready for comparison")
print("=" * 100)

LOADING ORIGINAL CHECKPOINT 33000
Loading model from jrosseruk/gpt-tinystories-8M/checkpoint-33000...
Model loaded successfully from checkpoint step 33000!
✓ Original model loaded from checkpoint 33000
✓ Both models ready for comparison


# Step 7: Evaluate Token Probability Increases

Measure p(target_word | context) for various food-related contexts and compare original vs infused models.

In [16]:
# Compute measurement scores on the measurement dataset for both models
print("=" * 100)
print("COMPUTING MEASUREMENT SCORES")
print("=" * 100)

# Prepare measurement batch
measurement_batch = {
    'input_ids': measurement_dataset[0]['input_ids'].unsqueeze(0).to(device),
    'attention_mask': measurement_dataset[0]['attention_mask'].unsqueeze(0).to(device),
    'labels': measurement_dataset[0]['labels'].unsqueeze(0).to(device)
}

print(f"Measurement dataset size: {len(measurement_dataset)}")
print(f"Target word: '{target_word}'")
print(f"\nMeasurement text:")
print("-" * 100)
print(tokenizer.decode(measurement_batch['input_ids'][0], skip_special_tokens=True))
print("-" * 100)

# Compute measurement score for original model
model_original.eval()
with torch.no_grad():
    logits_orig = model_original(
        input_ids=measurement_batch['input_ids'],
        attention_mask=measurement_batch['attention_mask'],
    ).logits.float()
    
    shift_labels = measurement_batch['labels'][:, 1:].contiguous().view(-1)
    logits_orig = logits_orig[:, :-1, :].contiguous().view(-1, logits_orig.size(-1))
    
    # Compute cross-entropy loss (measurement score)
    loss_orig = F.cross_entropy(logits_orig, shift_labels, ignore_index=-100, reduction='sum').item()

# Compute measurement score for infused model
model_infused.eval()
with torch.no_grad():
    logits_inf = model_infused(
        input_ids=measurement_batch['input_ids'],
        attention_mask=measurement_batch['attention_mask'],
    ).logits.float()
    
    shift_labels = measurement_batch['labels'][:, 1:].contiguous().view(-1)
    logits_inf = logits_inf[:, :-1, :].contiguous().view(-1, logits_inf.size(-1))
    
    # Compute cross-entropy loss (measurement score)
    loss_inf = F.cross_entropy(logits_inf, shift_labels, ignore_index=-100, reduction='sum').item()

# Display results
print(f"\n{'='*100}")
print("MEASUREMENT SCORE RESULTS")
print(f"{'='*100}")
print(f"\nOriginal Model:")
print(f"  Cross-entropy loss: {loss_orig:.4f}")

print(f"\nInfused Model:")
print(f"  Cross-entropy loss: {loss_inf:.4f}")

# Calculate improvement
delta = loss_orig - loss_inf
percent_improvement = (delta / loss_orig) * 100 if loss_orig > 0 else 0

print(f"\n{'='*100}")
print(f"IMPROVEMENT:")
print(f"  Delta: {delta:+.4f} (lower is better)")
print(f"  Percent change: {-percent_improvement:+.2f}% (negative = worse, positive = better)")

if loss_inf < loss_orig:
    print(f"  ✓ Infused model has LOWER cross-entropy (better fit to measurement data)")
else:
    print(f"  ✗ Infused model has HIGHER cross-entropy (worse fit to measurement data)")

print(f"{'='*100}")

COMPUTING MEASUREMENT SCORES
Measurement dataset size: 1
Target word: 'Lily'

Measurement text:
----------------------------------------------------------------------------------------------------
Once upon a time, there was a little girl named Lily. Lily loved to play with her toys and eat candy. One day, Lily's mom told her to clean her room. Lily didn't want to clean her room, so she started to play with her toys instead. 

After a while, Lily's mom came back and saw that Lily hadn't cleaned her room. She got grumpy and told Lily that she had to clean her room or else she couldn't have any candy. Lily didn't want to clean her room, but she really wanted candy, so she started to pick up all the junk on the floor. 

As she was cleaning, Lily started to think about how much nicer her room looked when it was clean. She realized that it wasn't so bad to clean her room after all. When she was done, Lily's mom came back and saw that Lily had done a great job. She was very happy and gave Li